In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path
from matplotlib.font_manager import FontProperties, fontManager

# Figure 6: Monthly comparison of Open-Meteo and BMKG observations

# Repository root
ROOT_DIR = Path.cwd()

if ROOT_DIR.name == "figure_scripts":
    ROOT_DIR = ROOT_DIR.parent

# Input data
DATA_FILE = (
    ROOT_DIR
    / "validation_monthly"
    / "bmkg_openmeteo_monthly_2025.csv"
)

# Output directory
OUTPUT_DIR = ROOT_DIR / "figure_scripts" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PNG_FILE = OUTPUT_DIR / "figure6_bmkg_openmeteo_comparison.png"
PDF_FILE = OUTPUT_DIR / "figure6_bmkg_openmeteo_comparison.pdf"

# Configure font
FONT_FILE = ROOT_DIR / "calibri.ttf"

if FONT_FILE.exists():
    fontManager.addfont(str(FONT_FILE))
    calibri = FontProperties(fname=str(FONT_FILE))
    plt.rcParams["font.family"] = calibri.get_name()
else:
    calibri = FontProperties(family="DejaVu Sans")
    plt.rcParams["font.family"] = "DejaVu Sans"

plt.rcParams["font.size"] = 16

# Load and prepare data
data = pd.read_csv(DATA_FILE)
data["month"] = pd.to_datetime(data["month"])
data = data.sort_values(["station_id", "month"])

stations = [
    (97120, "Majene"),
    (96109, "Pekanbaru"),
]

variables = [
    ("rainfall", "om_rainfall", "Rainfall", "Rainfall (mm/month)"),
    ("hum_avg", "om_hum_avg", "Mean relative humidity", "Relative humidity (%)"),
    ("temp_max", "om_temp_max", "Maximum temperature", "Temperature (°C)"),
    ("temp_min", "om_temp_min", "Minimum temperature", "Temperature (°C)"),
    ("temp_avg", "om_temp_avg", "Mean temperature", "Temperature (°C)"),
]

# Validate monthly observations
for station_id, station_name in stations:
    station_data = data[data["station_id"] == station_id]

    if len(station_data) != 12:
        raise ValueError(
            f"{station_name} (WMO {station_id}) has "
            f"{len(station_data)} rows; expected 12 monthly observations."
        )

    if station_data["month"].nunique() != 12:
        raise ValueError(
            f"{station_name} does not contain 12 unique months."
        )

# Create figure
fig, axes = plt.subplots(
    nrows=5,
    ncols=2,
    figsize=(12, 18),
    sharex=True,
)

for row, (bmkg_col, om_col, title, ylabel) in enumerate(variables):
    for col, (station_id, station_name) in enumerate(stations):
        ax = axes[row, col]
        station_data = data[data["station_id"] == station_id]

        ax.plot(
            station_data["month"],
            station_data[bmkg_col],
            marker="o",
            linewidth=1.5,
            markersize=4,
            label="BMKG",
        )

        ax.plot(
            station_data["month"],
            station_data[om_col],
            marker="s",
            linewidth=1.5,
            markersize=4,
            label="Open-Meteo",
        )

        ax.set_title(
            f"{title} | {station_name}",
            loc="left",
            fontsize=16,
            pad=8,
            fontproperties=calibri,
        )

        ax.set_ylabel(
            ylabel,
            fontproperties=calibri,
            fontsize=16,
        )

        ax.grid(
            axis="y",
            linestyle="--",
            linewidth=0.4,
            alpha=0.4,
        )

        ax.tick_params(axis="both", labelsize=14)

        for tick in ax.get_xticklabels() + ax.get_yticklabels():
            tick.set_fontproperties(calibri)
            tick.set_fontsize(14)

        if row == 0 and col == 0:
            ax.legend(
                loc="best",
                prop=FontProperties(
                    fname=str(FONT_FILE),
                    size=14
                ) if FONT_FILE.exists() else FontProperties(
                    family="DejaVu Sans",
                    size=14
                ),
                frameon=True,
            )

# Month labels: Jan, Feb, Mar, ... horizontally
for ax in axes[-1, :]:
    ax.set_xlabel("Month", fontproperties=calibri, fontsize=16)
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
    ax.tick_params(axis="x", labelrotation=0)

    for tick in ax.get_xticklabels():
        tick.set_fontproperties(calibri)
        tick.set_fontsize(14)
        tick.set_ha("center")

# Adjust spacing
plt.tight_layout(h_pad=1.2, w_pad=0.8)

# Save figures
fig.savefig(PNG_FILE, dpi=300, bbox_inches="tight")
fig.savefig(PDF_FILE, format="pdf", bbox_inches="tight")

plt.show()
plt.close(fig)

print("Figure 6 saved:")
print("PNG:", PNG_FILE)
print("PDF:", PDF_FILE)